# Práctica 1 — Dataset 1: Incidentes policiales de San Francisco (Bronze ➜ Silver)

**Objetivo:** Leer el CSV de incidentes desde la capa bronze (MinIO), limpiar/transformar con PySpark DataFrames y persistir en silver en formato Parquet.



In [1]:
# ============================================================
# 0) Imports + SparkSession
# ============================================================
from pyspark.sql import SparkSession, functions as F, types as T
import os, re, unicodedata

spark = SparkSession.builder.getOrCreate()

# Recomendación: fija la zona horaria de la sesión (evita sorpresas con timestamps)
spark.conf.set("spark.sql.session.timeZone", "UTC")

print("Spark version:", spark.version)

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /opt/spark/ivy/cache
The jars for the packages stored in: /opt/spark/ivy/jars
org.apache.spark#spark-hadoop-cloud_2.13 added as a dependency
io.delta#delta-spark_2.13 added as a dependency
org.apache.iceberg#iceberg-spark-runtime-4.0_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-381835e4-2c09-4a8d-aa28-e7688df1f5df;1.0
	confs: [default]
	found org.apache.spark#spark-hadoop-cloud_2.13;4.0.1 in central
	found org.apache.hadoop#hadoop-client-runtime;3.4.1 in central
	found org.apache.hadoop#hadoop-client-api;3.4.1 in central
	found org.xerial.snappy#snappy-java;1.1.10.7 in central
	found org.slf4j#slf4j-api;2.0.16 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.hadoop#hadoop-aws;3.4.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.2.5.Final in central
	found s

Spark version: 4.0.1


In [2]:
# Funcion para facilitar visualización
from IPython.display import display
import pandas as pd
pd.set_option("display.max_columns", None)  # muestra todas las columnas

def display_df(
    df,
    n=10,
    columns=None,
    sort_by=None,
    ascending=False
):
    """
    Visualiza un DataFrame de Spark en Jupyter de forma similar a Databricks display().

    Parámetros:
    - df: DataFrame de Spark
    - n: número máximo de filas a mostrar (default: 20)
    - columns: lista opcional de columnas a mostrar
    - sort_by: columna opcional para ordenar
    - ascending: orden ascendente o descendente
    """

    if columns:
        df = df.select(columns)

    if sort_by:
        df = df.orderBy(sort_by, ascending=ascending)

    pdf = df.limit(n).toPandas()
    display(pdf)


In [3]:
# ============================================================
# 0.1) Configuración de acceso a MinIO
# ============================================================
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "http://minio:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY", "minioadmin")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY", "minioadmin")
MINIO_USE_SSL = os.getenv("MINIO_USE_SSL", "false").lower() == "true"

hconf = spark._jsc.hadoopConfiguration()
hconf.set("fs.s3a.endpoint", MINIO_ENDPOINT)
hconf.set("fs.s3a.access.key", MINIO_ACCESS_KEY)
hconf.set("fs.s3a.secret.key", MINIO_SECRET_KEY)
hconf.set("fs.s3a.path.style.access", "true")
hconf.set("fs.s3a.connection.ssl.enabled", "true" if MINIO_USE_SSL else "false")
hconf.set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
hconf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

print("MINIO_ENDPOINT =", MINIO_ENDPOINT)
print("MINIO_ACCESS_KEY =", MINIO_ACCESS_KEY)

MINIO_ENDPOINT = http://minio:9000
MINIO_ACCESS_KEY = minioadmin


## 1) Lectura desde Bronze (sin inferir esquema)

- Se lee el dataset desde la carpeta en **bronze**.
- No se infiere el esquema (`inferSchema = false`), por lo que inicialmente las columnas vendrán definidas como string.

In [4]:
BRONZE_PATH = "s3a://bronze/Police_Department_Incident_Reports__2018_to_Present_20251208.csv/"

df_raw = (
    spark.read
        .option("header", True)
        .option("inferSchema", "false")
        .csv(BRONZE_PATH)
)

df_raw.printSchema()
display_df(df_raw, n=5)


root
 |-- Row ID: string (nullable = true)
 |-- Incident Datetime: string (nullable = true)
 |-- Incident Date: string (nullable = true)
 |-- Incident Time: string (nullable = true)
 |-- Incident Year: string (nullable = true)
 |-- Incident Day of Week: string (nullable = true)
 |-- Report Datetime: string (nullable = true)
 |-- Incident ID: string (nullable = true)
 |-- Incident Number: string (nullable = true)
 |-- CAD Number: string (nullable = true)
 |-- Report Type Code: string (nullable = true)
 |-- Report Type Description: string (nullable = true)
 |-- Filed Online: string (nullable = true)
 |-- Incident Code: string (nullable = true)
 |-- Incident Category: string (nullable = true)
 |-- Incident Subcategory: string (nullable = true)
 |-- Incident Description: string (nullable = true)
 |-- Resolution: string (nullable = true)
 |-- Intersection: string (nullable = true)
 |-- CNN: string (nullable = true)
 |-- Police District: string (nullable = true)
 |-- Analysis Neighborhood: s

26/03/03 11:21:21 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,Row ID,Incident Datetime,Incident Date,Incident Time,Incident Year,Incident Day of Week,Report Datetime,Incident ID,Incident Number,CAD Number,Report Type Code,Report Type Description,Filed Online,Incident Code,Incident Category,Incident Subcategory,Incident Description,Resolution,Intersection,CNN,Police District,Analysis Neighborhood,Supervisor District,Supervisor District 2012,Latitude,Longitude,Point,data_as_of,data_loaded_at
0,152278216710,2025/10/31 05:55:00 PM,2025/10/31,17:55,2025,Friday,2025/10/31 05:55:00 PM,1522782,250613625,253042598,II,Initial,None,16710,Drug Offense,Drug Violation,"Narcotics Paraphernalia, Possession of",Cite or Arrest Adult,24TH ST \ LILAC ST,24074000,Mission,Mission,9,9,37.75226974487,-122.41787719727,POINT (-122.417877197 37.752269745),2025/11/04 09:37:41 AM,2025/11/05 09:59:28 AM
1,148998204134,2025/06/13 12:41:00 PM,2025/06/13,12:41,2025,Friday,2025/06/13 12:46:00 PM,1489982,250329888,251641497,II,Initial,None,04134,Assault,Simple Assault,Battery,Open or Active,JOHN F SHELLEY DR \ MANSELL ST,20872000,Ingleside,McLaren Park,9,9,37.71812820435,-122.41417694092,POINT (-122.414176941 37.718128204),2025/06/14 09:37:39 AM,2025/06/15 09:53:25 AM
2,152328663010,2025/11/03 02:00:00 PM,2025/11/03,14:00,2025,Monday,2025/11/03 02:00:00 PM,1523286,250269557,None,IS,Initial Supplement,None,63010,Warrant,Other,"Warrant Arrest, Local SF Warrant",Cite or Arrest Adult,None,None,Out of SF,None,None,None,None,None,None,2025/11/04 09:37:41 AM,2025/11/05 09:59:28 AM
3,149000206224,2025/05/21 12:00:00 AM,2025/05/21,00:00,2025,Wednesday,2025/05/21 08:15:00 AM,1490002,256057940,None,II,Coplogic Initial,true,06224,Larceny Theft,Larceny - From Vehicle,"Theft, From Unlocked Vehicle, >$950",Open or Active,None,None,Ingleside,None,None,None,None,None,None,2025/06/14 09:37:39 AM,2025/06/15 09:53:25 AM
4,152326906304,2025/10/23 12:00:00 AM,2025/10/23,00:00,2025,Thursday,2025/11/03 01:24:00 PM,1523269,250618885,253071894,II,Initial,None,06304,Larceny Theft,Larceny Theft - From Building,"Theft, From Building, >$950",Open or Active,38TH AVE \ CABRILLO ST,27865000,Richmond,Outer Richmond,1,1,37.77379989624,-122.49825286865,POINT (-122.498252869 37.773799896),2025/11/04 09:37:41 AM,2025/11/05 09:59:28 AM


## 2) Selección de columnas requeridas

Seleccionamos **solo** las columnas pedidas en el enunciado.

In [5]:
required_cols = [
    "Incident ID",
    "Incident Number",
    "Row ID",
    "Incident Datetime",
    "Report Datetime",
    "Incident Code",
    "Incident Category",
    "Incident Subcategory",
    "Incident Description",
    "Report Type Code",
    "Resolution",
    "Police District",
    "Analysis Neighborhood",
    "Latitude",
    "Longitude",
]

df_selected = df_raw.select(*required_cols)

df_selected.printSchema()
display_df(df_selected, n=5)


root
 |-- Incident ID: string (nullable = true)
 |-- Incident Number: string (nullable = true)
 |-- Row ID: string (nullable = true)
 |-- Incident Datetime: string (nullable = true)
 |-- Report Datetime: string (nullable = true)
 |-- Incident Code: string (nullable = true)
 |-- Incident Category: string (nullable = true)
 |-- Incident Subcategory: string (nullable = true)
 |-- Incident Description: string (nullable = true)
 |-- Report Type Code: string (nullable = true)
 |-- Resolution: string (nullable = true)
 |-- Police District: string (nullable = true)
 |-- Analysis Neighborhood: string (nullable = true)
 |-- Latitude: string (nullable = true)
 |-- Longitude: string (nullable = true)



,Incident ID,Incident Number,Row ID,Incident Datetime,Report Datetime,Incident Code,Incident Category,Incident Subcategory,Incident Description,Report Type Code,Resolution,Police District,Analysis Neighborhood,Latitude,Longitude
0,1522782,250613625,152278216710,2025/10/31 05:55:00 PM,2025/10/31 05:55:00 PM,16710,Drug Offense,Drug Violation,"Narcotics Paraphernalia, Possession of",II,Cite or Arrest Adult,Mission,Mission,37.75226974487,-122.41787719727
1,1489982,250329888,148998204134,2025/06/13 12:41:00 PM,2025/06/13 12:46:00 PM,04134,Assault,Simple Assault,Battery,II,Open or Active,Ingleside,McLaren Park,37.71812820435,-122.41417694092
2,1523286,250269557,152328663010,2025/11/03 02:00:00 PM,2025/11/03 02:00:00 PM,63010,Warrant,Other,"Warrant Arrest, Local SF Warrant",IS,Cite or Arrest Adult,Out of SF,None,None,None
3,1490002,256057940,149000206224,2025/05/21 12:00:00 AM,2025/05/21 08:15:00 AM,06224,Larceny Theft,Larceny - From Vehicle,"Theft, From Unlocked Vehicle, >$950",II,Open or Active,Ingleside,None,None,None
4,1523269,250618885,152326906304,2025/10/23 12:00:00 AM,2025/11/03 01:24:00 PM,06304,Larceny Theft,Larceny Theft - From Building,"Theft, From Building, >$950",II,Open or Active,Richmond,Outer Richmond,37.77379989624,-122.49825286865


## 3) Enfoque alternativo: eliminar columnas que no necesitas

En el paso 2 hemos usado `select(...)` para quedarnos con las columnas necesarias.

Una alternativa equivalente es eliminar las columnas sobrantes con `drop(...)`.
Esto es útil cuando tienes claro qué columnas no quieres (por ejemplo, una lista larga)
o cuando quieres mantener el orden original del dataset, pero sin campos innecesarios.

Documentación oficial (Spark 4.0.1):  
- `pyspark.sql.DataFrame.drop`: https://spark.apache.org/docs/4.0.1/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.drop.html

In [6]:
cols_to_drop = [c for c in df_raw.columns if c not in required_cols]

df_alt = df_raw.drop(*cols_to_drop)

print("Mismas columnas (como conjunto):", set(df_alt.columns) == set(df_selected.columns))
print("Nº columnas df_selected:", len(df_selected.columns))
print("Nº columnas df_alt     :", len(df_alt.columns))

display_df(df_alt, n=5)


Mismas columnas (como conjunto): True
Nº columnas df_selected: 15
Nº columnas df_alt     : 15


,Row ID,Incident Datetime,Report Datetime,Incident ID,Incident Number,Report Type Code,Incident Code,Incident Category,Incident Subcategory,Incident Description,Resolution,Police District,Analysis Neighborhood,Latitude,Longitude
0,152278216710,2025/10/31 05:55:00 PM,2025/10/31 05:55:00 PM,1522782,250613625,II,16710,Drug Offense,Drug Violation,"Narcotics Paraphernalia, Possession of",Cite or Arrest Adult,Mission,Mission,37.75226974487,-122.41787719727
1,148998204134,2025/06/13 12:41:00 PM,2025/06/13 12:46:00 PM,1489982,250329888,II,04134,Assault,Simple Assault,Battery,Open or Active,Ingleside,McLaren Park,37.71812820435,-122.41417694092
2,152328663010,2025/11/03 02:00:00 PM,2025/11/03 02:00:00 PM,1523286,250269557,IS,63010,Warrant,Other,"Warrant Arrest, Local SF Warrant",Cite or Arrest Adult,Out of SF,None,None,None
3,149000206224,2025/05/21 12:00:00 AM,2025/05/21 08:15:00 AM,1490002,256057940,II,06224,Larceny Theft,Larceny - From Vehicle,"Theft, From Unlocked Vehicle, >$950",Open or Active,Ingleside,None,None,None
4,152326906304,2025/10/23 12:00:00 AM,2025/11/03 01:24:00 PM,1523269,250618885,II,06304,Larceny Theft,Larceny Theft - From Building,"Theft, From Building, >$950",Open or Active,Richmond,Outer Richmond,37.77379989624,-122.49825286865


## 4) Renombrado a `snake_case`

Reglas:
- minúsculas
- palabras separadas por `_`
- sin espacios ni caracteres especiales

Ejemplo: `Incident Datetime` → `incident_datetime`

In [7]:
def to_snake_case(name: str):
    # Normaliza acentos y caracteres raros
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("ascii")
    # Sustituye no-alfanumérico por espacio
    name = re.sub(r"[^0-9a-zA-Z]+", " ", name)
    # Separa camel case (por si acaso) y colapsa espacios
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1 \2", name)
    name = re.sub(r"\s+", " ", name).strip().lower()
    # Une con underscore
    return name.replace(" ", "_")

snake_cols = [to_snake_case(c) for c in df_selected.columns]
df = df_selected.toDF(*snake_cols)

df.printSchema()
display_df(df, n=5)


root
 |-- incident_id: string (nullable = true)
 |-- incident_number: string (nullable = true)
 |-- row_id: string (nullable = true)
 |-- incident_datetime: string (nullable = true)
 |-- report_datetime: string (nullable = true)
 |-- incident_code: string (nullable = true)
 |-- incident_category: string (nullable = true)
 |-- incident_subcategory: string (nullable = true)
 |-- incident_description: string (nullable = true)
 |-- report_type_code: string (nullable = true)
 |-- resolution: string (nullable = true)
 |-- police_district: string (nullable = true)
 |-- analysis_neighborhood: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)



,incident_id,incident_number,row_id,incident_datetime,report_datetime,incident_code,incident_category,incident_subcategory,incident_description,report_type_code,resolution,police_district,analysis_neighborhood,latitude,longitude
0,1522782,250613625,152278216710,2025/10/31 05:55:00 PM,2025/10/31 05:55:00 PM,16710,Drug Offense,Drug Violation,"Narcotics Paraphernalia, Possession of",II,Cite or Arrest Adult,Mission,Mission,37.75226974487,-122.41787719727
1,1489982,250329888,148998204134,2025/06/13 12:41:00 PM,2025/06/13 12:46:00 PM,04134,Assault,Simple Assault,Battery,II,Open or Active,Ingleside,McLaren Park,37.71812820435,-122.41417694092
2,1523286,250269557,152328663010,2025/11/03 02:00:00 PM,2025/11/03 02:00:00 PM,63010,Warrant,Other,"Warrant Arrest, Local SF Warrant",IS,Cite or Arrest Adult,Out of SF,None,None,None
3,1490002,256057940,149000206224,2025/05/21 12:00:00 AM,2025/05/21 08:15:00 AM,06224,Larceny Theft,Larceny - From Vehicle,"Theft, From Unlocked Vehicle, >$950",II,Open or Active,Ingleside,None,None,None
4,1523269,250618885,152326906304,2025/10/23 12:00:00 AM,2025/11/03 01:24:00 PM,06304,Larceny Theft,Larceny Theft - From Building,"Theft, From Building, >$950",II,Open or Active,Richmond,Outer Richmond,37.77379989624,-122.49825286865


## 5) Convertir `incident_datetime` y `report_datetime` a `timestamp`

Como el CSV puede venir con formatos distintos (dependiendo de la exportación),
usamos un parseo “robusto” intentando varios patrones y quedándonos con el primero que funcione.

In [8]:
def parse_timestamp(col):
    # Usamos try_to_timestamp para tolerar múltiples formatos incluso con ANSI mode activo.
    # Si un formato no encaja, devuelve NULL en lugar de lanzar excepción.
    return F.coalesce(
        F.try_to_timestamp(col, F.lit("yyyy-MM-dd'T'HH:mm:ss.SSSX")),
        F.try_to_timestamp(col, F.lit("yyyy-MM-dd'T'HH:mm:ss.SSS")),
        F.try_to_timestamp(col, F.lit("yyyy-MM-dd'T'HH:mm:ssX")),
        F.try_to_timestamp(col, F.lit("yyyy-MM-dd'T'HH:mm:ss")),
        F.try_to_timestamp(col, F.lit("yyyy-MM-dd HH:mm:ss")),
        # Formatos observados con barras y AM/PM, p.ej. 2025/10/31 05:55:00 PM
        F.try_to_timestamp(col, F.lit("yyyy/MM/dd hh:mm:ss a")),
        F.try_to_timestamp(col, F.lit("yyyy/MM/dd HH:mm:ss")),
        # Fallback al parser por defecto
        F.try_to_timestamp(col)
    )

df = (
    df.withColumn("incident_datetime", parse_timestamp(F.col("incident_datetime")))
      .withColumn("report_datetime", parse_timestamp(F.col("report_datetime")))
)

df.printSchema()
display_df(df, n=5)


root
 |-- incident_id: string (nullable = true)
 |-- incident_number: string (nullable = true)
 |-- row_id: string (nullable = true)
 |-- incident_datetime: timestamp (nullable = true)
 |-- report_datetime: timestamp (nullable = true)
 |-- incident_code: string (nullable = true)
 |-- incident_category: string (nullable = true)
 |-- incident_subcategory: string (nullable = true)
 |-- incident_description: string (nullable = true)
 |-- report_type_code: string (nullable = true)
 |-- resolution: string (nullable = true)
 |-- police_district: string (nullable = true)
 |-- analysis_neighborhood: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)



,incident_id,incident_number,row_id,incident_datetime,report_datetime,incident_code,incident_category,incident_subcategory,incident_description,report_type_code,resolution,police_district,analysis_neighborhood,latitude,longitude
0,1522782,250613625,152278216710,2025-10-31 17:55:00,2025-10-31 17:55:00,16710,Drug Offense,Drug Violation,"Narcotics Paraphernalia, Possession of",II,Cite or Arrest Adult,Mission,Mission,37.75226974487,-122.41787719727
1,1489982,250329888,148998204134,2025-06-13 12:41:00,2025-06-13 12:46:00,04134,Assault,Simple Assault,Battery,II,Open or Active,Ingleside,McLaren Park,37.71812820435,-122.41417694092
2,1523286,250269557,152328663010,2025-11-03 14:00:00,2025-11-03 14:00:00,63010,Warrant,Other,"Warrant Arrest, Local SF Warrant",IS,Cite or Arrest Adult,Out of SF,None,None,None
3,1490002,256057940,149000206224,2025-05-21 00:00:00,2025-05-21 08:15:00,06224,Larceny Theft,Larceny - From Vehicle,"Theft, From Unlocked Vehicle, >$950",II,Open or Active,Ingleside,None,None,None
4,1523269,250618885,152326906304,2025-10-23 00:00:00,2025-11-03 13:24:00,06304,Larceny Theft,Larceny Theft - From Building,"Theft, From Building, >$950",II,Open or Active,Richmond,Outer Richmond,37.77379989624,-122.49825286865


## 6) Convertir el resto de columnas numéricas a tipos adecuados

Convertimos identificadores y coordenadas a tipos numéricos soportados por Spark.

In [9]:
df = (
    df.withColumn("incident_id", F.col("incident_id").cast(T.LongType()))
      .withColumn("incident_number", F.col("incident_number").cast(T.LongType()))
      .withColumn("row_id", F.col("row_id").cast(T.LongType()))
      .withColumn("incident_code", F.col("incident_code").cast(T.IntegerType()))
      .withColumn("latitude", F.col("latitude").cast(T.DoubleType()))
      .withColumn("longitude", F.col("longitude").cast(T.DoubleType()))
)

df.printSchema()
display_df(df, n=5)


root
 |-- incident_id: long (nullable = true)
 |-- incident_number: long (nullable = true)
 |-- row_id: long (nullable = true)
 |-- incident_datetime: timestamp (nullable = true)
 |-- report_datetime: timestamp (nullable = true)
 |-- incident_code: integer (nullable = true)
 |-- incident_category: string (nullable = true)
 |-- incident_subcategory: string (nullable = true)
 |-- incident_description: string (nullable = true)
 |-- report_type_code: string (nullable = true)
 |-- resolution: string (nullable = true)
 |-- police_district: string (nullable = true)
 |-- analysis_neighborhood: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)



,incident_id,incident_number,row_id,incident_datetime,report_datetime,incident_code,incident_category,incident_subcategory,incident_description,report_type_code,resolution,police_district,analysis_neighborhood,latitude,longitude
0,1522782,250613625,152278216710,2025-10-31 17:55:00,2025-10-31 17:55:00,16710,Drug Offense,Drug Violation,"Narcotics Paraphernalia, Possession of",II,Cite or Arrest Adult,Mission,Mission,37.752270,-122.417877
1,1489982,250329888,148998204134,2025-06-13 12:41:00,2025-06-13 12:46:00,4134,Assault,Simple Assault,Battery,II,Open or Active,Ingleside,McLaren Park,37.718128,-122.414177
2,1523286,250269557,152328663010,2025-11-03 14:00:00,2025-11-03 14:00:00,63010,Warrant,Other,"Warrant Arrest, Local SF Warrant",IS,Cite or Arrest Adult,Out of SF,None,NaN,NaN
3,1490002,256057940,149000206224,2025-05-21 00:00:00,2025-05-21 08:15:00,6224,Larceny Theft,Larceny - From Vehicle,"Theft, From Unlocked Vehicle, >$950",II,Open or Active,Ingleside,None,NaN,NaN
4,1523269,250618885,152326906304,2025-10-23 00:00:00,2025-11-03 13:24:00,6304,Larceny Theft,Larceny Theft - From Building,"Theft, From Building, >$950",II,Open or Active,Richmond,Outer Richmond,37.773800,-122.498253


## 7) Crear `reporting_delay_minutes`

Diferencia en minutos entre `report_datetime` e `incident_datetime`.

In [10]:
df = df.withColumn(
    "reporting_delay_minutes",
    ((F.col("report_datetime").cast("long") - F.col("incident_datetime").cast("long")) / 60).cast("int")
)

display_df(df.select("incident_datetime", "report_datetime", "reporting_delay_minutes"), n=10)


,incident_datetime,report_datetime,reporting_delay_minutes
0,2025-10-31 17:55:00,2025-10-31 17:55:00,0
1,2025-06-13 12:41:00,2025-06-13 12:46:00,5
2,2025-11-03 14:00:00,2025-11-03 14:00:00,0
3,2025-05-21 00:00:00,2025-05-21 08:15:00,495
4,2025-10-23 00:00:00,2025-11-03 13:24:00,16644
5,2025-06-11 08:00:00,2025-06-12 11:27:00,1647
6,2025-06-12 00:09:00,2025-06-12 00:09:00,0
7,2025-05-23 00:00:00,2025-06-12 12:24:00,29544
8,2025-06-13 02:55:00,2025-06-13 02:55:00,0
9,2025-06-11 22:05:00,2025-06-11 22:09:00,4


## 8) Crear `delay_bucket` (categoría por rango de minutos)

Rangos requeridos (valores **string**):
- `<0`
- `0_10`
- `10_60`
- `60_1440`
- `>1440`

In [11]:
delay = F.col("reporting_delay_minutes")

df = df.withColumn(
    "delay_bucket",
    F.when(delay.isNull(), F.lit(None).cast("string"))
     .when(delay < 0, F.lit("<0"))
     .when((delay >= 0) & (delay < 10), F.lit("0_10"))
     .when((delay >= 10) & (delay < 60), F.lit("10_60"))
     .when((delay >= 60) & (delay <= 1440), F.lit("60_1440"))
     .otherwise(F.lit(">1440"))
)

display_df(df.groupBy("delay_bucket").count().orderBy("delay_bucket"), n=20)


,delay_bucket,count
0,0_10,305404
1,10_60,120615
2,60_1440,299510
3,<0,8
4,>1440,263122


## 9) Escritura en Silver (Parquet)

Persistimos el DataFrame final en:

- `silver/sf_police_incidents/`

usando formato **Parquet**.

In [12]:
SILVER_PATH = "s3a://silver/sf_police_incidents/"

(
    df.write
      .mode("overwrite")
      .parquet(SILVER_PATH)
)


df_check = spark.read.parquet(SILVER_PATH)
print("Filas en silver:", df_check.count())
df_check.printSchema()

26/03/03 11:21:34 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/03 11:21:34 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/03 11:21:34 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/03 11:21:34 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/03 11:21:34 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/03 11:21:44 WARN S3ABlockOutputStream: Application invoked the Syncable API against stream writing to sf_police_incidents/_temporary/0/_temporary/attempt_202603031121332101245017950112465_0011_m_000011_32/part-00011-

Filas en silver: 988659
root
 |-- incident_id: long (nullable = true)
 |-- incident_number: long (nullable = true)
 |-- row_id: long (nullable = true)
 |-- incident_datetime: timestamp (nullable = true)
 |-- report_datetime: timestamp (nullable = true)
 |-- incident_code: integer (nullable = true)
 |-- incident_category: string (nullable = true)
 |-- incident_subcategory: string (nullable = true)
 |-- incident_description: string (nullable = true)
 |-- report_type_code: string (nullable = true)
 |-- resolution: string (nullable = true)
 |-- police_district: string (nullable = true)
 |-- analysis_neighborhood: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- reporting_delay_minutes: integer (nullable = true)
 |-- delay_bucket: string (nullable = true)



## 10) Preguntas analíticas

### i) ¿Cuántas filas tienen valores nulos en `latitude` o `longitude`?

In [13]:
null_latlon_rows = df.filter(F.col("latitude").isNull() | F.col("longitude").isNull()).count()
print(null_latlon_rows)

[Stage 16:===================================================>    (11 + 1) / 12]

54725


Hay 54725 filas con valores nulos en latitud o longitud. Esto corresponde a filas sin geolocalización completa; en análisis geoespacial conviene filtrarlas o tratarlas como "sin localización". Por lo tanto, es común que estos valores sean nulos si la información original venía incompleta.

### ii) ¿Cuántos `incident_id` aparecen más de una vez en el dataset?

Se tiene en cuenta que un mismo incidente puede aparecer en varias filas.

In [14]:
duplicated_incident_ids = (
    df.groupBy("incident_id")
      .count()
      .filter(F.col("count") > 1)
      .count()
)

print(duplicated_incident_ids)

[Stage 19:>                                                       (0 + 12) / 12]

127111


Un mismo incident_id puede estar desglosado en varias filas en nuestro caso son 127111 numeros de incident_id que aparecen mas de una vez; lo que hicimos fue agrupar por incident_id, contar ocurrencias, filtrar los que tienen más de 1, y finalmente contar cuántos IDs cumplen esa condición.

### iii) ¿Cuántos incidentes tienen un retraso entre 10 y 60 minutos en su procesamiento?

Usando exclusivamente la columna `delay_bucket`.

In [16]:
#Con repetición de IDs.
rows_10_60 = df.filter(F.col("delay_bucket") == "10_60").count()

#Sin repeticion de IDs.
incidents_10_60 = (
    df.filter(F.col("delay_bucket") == "10_60")
      .select("incident_id")
      .distinct()
      .count()
)

print(rows_10_60)
print(incidents_10_60)

[Stage 28:===================================================>    (11 + 1) / 12]

120615
92823


Hay 92.823 incidentes únicos que tienen un retraso entre 10 y 60 minutos en su procesamiento (obtenido al contar los incident_id distintos filtrando por delay_bucket == "10_60"). Si contabilizamos el volumen total de registros (filas) en este rango, el número es de 120.615. Esta diferencia nos indica que algunos incidentes generan múltiples registros, actualizaciones o entradas asociadas a ese mismo tramo de retraso.